# Frozen confidence-dependent recognition score

This small synthetic example applies the bundled score without fitting. The score is
`S = (1 − .41q − .22(1−q)) ipTM + .41q B + .22(1−q) M`, where B is recognition
Betti fraction and M is mock-calibrated contact enrichment. **q uses cohort mean
pLDDT**, on the 0–100 generator scale, against five frozen reference means.
The individual ipTM remains a score component. These synthetic rows demonstrate the API;
they are not measured performance or mock controls. Install the release and select its Python kernel.

In [ ]:
# Report the active kernel and package versions.
import sys, json, time
from pathlib import Path
from importlib.metadata import version
from importlib.resources import files
import numpy as np
import polars as pl
from tcren.contacts.motif_optimize import apply_confidence_score
started = time.monotonic()
print({name: version(name) for name in ["tcren", "numpy", "polars"]})
print("Python", sys.version.split()[0])
resource_path = files("tcren.data").joinpath("motif_confidence_reference.json")
resource = json.loads(resource_path.read_text())


In [ ]:
# Construct two synthetic unlabelled cohorts with already computed components.
rows = pl.DataFrame({
    "id": ["example-1", "example-2", "example-3", "example-4"],
    "dataset": ["example"] * 4, "stratum": ["all"] * 4,
    "epitope": ["cohort-a", "cohort-a", "cohort-b", "cohort-b"],
    "iptm": [.72, .80, .73, .84], "betti": [.30, .42, .33, .45],
    "enrichment": [.60, .81, .52, .78], "plddt": [91., 93., 94., 96.],
})
scored = apply_confidence_score(rows, resource)
scored.select("id", "score", "gate_q", "w_I", "w_B", "w_M")

In [ ]:
# Check bounds, weight sums and exact row-order invariance.
assert scored["score"].is_between(0, 1).all()
assert np.allclose(scored.select(pl.sum_horizontal("w_I", "w_B", "w_M")).to_numpy(), 1)
reverse = apply_confidence_score(rows.reverse(), resource).sort("id")
assert np.allclose(scored.sort("id")["score"], reverse["score"], rtol=0, atol=1e-15)
print("Bounds, weight sums and row-order checks passed")

## Structures and small batches

For one complete epitope/pMHC cohort, supply a confidence TSV with
`complex.id`, `iptm`, and generator `plddt` (0–100):

```sh
tcren screen -s models.tar.gz --confidence confidence.tsv -t 0 -o scores.tsv
```

Run each epitope/pMHC cohort separately; do not submit mixed epitopes as one cohort.
The default minimum is 30 contact-bearing models; zero-contact rows retain reasons
and no score. Recover missing generator pLDDT with the documented
`motif-confidence-inputs` source/hash protocol when reproducing recorded results;
omission otherwise invokes the frozen missing-value policy.
The CLI owns contact extraction, bundled mock calibration and scoring; do not implement
those stages in a notebook. Submit the entire intended unlabelled cohort, including rows
with unknown binding outcomes. An isolated structure supplies a different confidence
context; a proxy must be explicitly selected and reported. Experimental B-factors are not pLDDT.

The weights were selected in TCRvdb development, and the empirical confidence reference
contains five cohort means. The score is bounded, not a calibrated binding probability.

In [ ]:
# Report completion of this deterministic, no-fit example.
print({"rows": scored.height, "seconds": round(time.monotonic() - started, 3), "fitting": False})